In [1]:
"""
additional_LST_analysis.py

Additional analysis for reviewer response: Land Surface Temperature (LST),
residential energy consumption per capita (REPC), and landscape metrics.

Main workflow:
1. Set project structure.
2. Merge the cleaned modelling CSV with the filtered neighborhood shapefile.
3. Report possible shapefile field-name truncation caused by the 10-character limit.
4. Calculate neighborhood-level median LST from the full LST raster.
5. Calculate Spearman correlation between REPC and LST for all neighborhoods and by urbanity level.
6. Calculate neighborhood-level median LST for selected land-cover classes.
7. Calculate Spearman correlations between class-specific LST and related landscape metrics.

Required input files:
- model/form_energy_final_cleaned.csv
- shp/10580_filtered_neighborhood_boundariesx.shp
- data/NL_LST_2022_winter_m_3.tif
- data/Extract_NL_water3.tif
- data/Extract_NL_built3.tif
- data/Extract_NL_Dense_Short3.tif
- data/Extract_NL_Dense_Tall3.tif

Main outputs:
- shp/merged_cleaned_shapefile.shp
- model/Spearman_REPC_LST_by_ste_mvs_and_all.csv
- model/LST_landcover_metric_spearman_summary.csv
- model/LST_landcover_metric_spearman_grouped.csv
"""

import os
import warnings
from typing import Dict, List, Tuple

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterstats import zonal_stats

warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 0. PROJECT STRUCTURE
# ============================================================

BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "output")
WEIGHT_DIR = os.path.join(BASE_DIR, "weights")
SRC_DIR = os.path.join(BASE_DIR, "src")
SHP_DIR = os.path.join(BASE_DIR, "shp")
MODEL_DIR = os.path.join(BASE_DIR, "model")
FIGURE_DIR = os.path.join(BASE_DIR, "figures")

for directory in [DATA_DIR, OUT_DIR, WEIGHT_DIR, SRC_DIR, SHP_DIR, MODEL_DIR, FIGURE_DIR]:
    os.makedirs(directory, exist_ok=True)

print("PROJECT STRUCTURE READY")
print("BASE_DIR:", BASE_DIR)


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

CLEANED_CSV = os.path.join(MODEL_DIR, "form_energy_final_cleaned.csv")
FILTERED_SHP = os.path.join(SHP_DIR, "10580_filtered_neighborhood_boundariesx.shp")
MERGED_SHP = os.path.join(SHP_DIR, "merged_cleaned_shapefile.shp")

COMM_ID_COL = "buurtcode"
STE_COL = "ste_mvs"

# The original CSV column is likely 'ec_inten_percapita'.
# After saving to shapefile, the name may be truncated to 'ec_inten_p'.
REPC_CANDIDATES = ["ec_inten_percapita", "ec_inten_p"]

FULL_LST_TIF = os.path.join(DATA_DIR, "NL_LST_2022_winter_m_3.tif")
FULL_LST_FIELD = "LST_median_2022"

# Class-specific LST rasters and related landscape metrics.
# Some metric names are truncated because shapefile field names are limited to 10 characters.
LST_TIF_NAME = {
    "water": "Extract_NL_water3.tif",
    "built": "Extract_NL_built3.tif",
    "dense_short": "Extract_NL_Dense_Short3.tif",
    "dense_tall": "Extract_NL_Dense_Tall3.tif",
}

PRIMARY_METRIC = {
    "water": "AI_water",
    "built": "AI_built",
    "dense_short": "PLAND_Dens",  # truncated from PLAND_Dense_Short
    "dense_tall": "PLAND_De_1",   # truncated from PLAND_Dense_Tall
}

SECONDARY_METRIC = {
    "water": "IJI_water",
    "built": "IJI_built",
    "dense_short": "AI_Dense_S",  # truncated from AI_Dense_Short
    "dense_tall": "AI_Dense_T",   # truncated from AI_Dense_Tall
}

LAND_COVER_TYPES = ["water", "built", "dense_short", "dense_tall"]


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def require_file(path: str, label: str) -> None:
    """Raise an error if a required file does not exist."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required {label} not found: {path}")


def find_existing_column(df: pd.DataFrame, candidates: List[str], label: str) -> str:
    """Return the first existing column from a list of candidate names."""
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Could not find {label}. Tried: {candidates}")


def truncate_shapefile_field(name: str) -> str:
    """Return the 10-character shapefile version of a field name."""
    return str(name)[:10]


def print_shapefile_truncation_report(csv_cols: List[str], shp_cols: List[str]) -> None:
    """Print possible mapping between original CSV names and truncated shapefile names."""
    mapping: Dict[str, List[str]] = {}
    for col in csv_cols:
        truncated = truncate_shapefile_field(col)
        mapping.setdefault(truncated, []).append(col)

    print("\n=== Possible Truncation Mapping (shapefile field → CSV field) ===")
    for shp_col in shp_cols:
        if shp_col in mapping:
            print(f"{shp_col}  ←  {mapping[shp_col]}")


def merge_csv_with_shapefile(
    csv_path: str = CLEANED_CSV,
    shp_path: str = FILTERED_SHP,
    output_path: str = MERGED_SHP
) -> gpd.GeoDataFrame:
    """Merge cleaned modelling CSV into the filtered neighborhood shapefile."""
    require_file(csv_path, "cleaned model CSV")
    require_file(shp_path, "filtered shapefile")

    df = pd.read_csv(csv_path, dtype={COMM_ID_COL: str})
    gdf = gpd.read_file(shp_path)
    gdf[COMM_ID_COL] = gdf[COMM_ID_COL].astype(str)

    print("\nLoaded successfully.")
    print("Rows in CSV:", len(df))
    print("Rows in SHP:", len(gdf))

    print("\n=== Loaded CSV Columns ===")
    print(df.columns.tolist())

    print("\n=== Loaded SHP Columns ===")
    print(gdf.columns.tolist())

    if COMM_ID_COL not in df.columns:
        raise ValueError(f"CSV missing column: '{COMM_ID_COL}'")
    if COMM_ID_COL not in gdf.columns:
        raise ValueError(f"Shapefile missing column: '{COMM_ID_COL}'")

    merged = gdf.merge(df, on=COMM_ID_COL, how="inner")

    print("\n=== Merged GeoDataFrame Preview ===")
    print(merged.head())

    original_shp_cols = set(gdf.columns)
    original_csv_cols = set(df.columns)
    merged_cols = set(merged.columns)

    new_columns = merged_cols - original_shp_cols
    shared_columns = original_csv_cols & original_shp_cols

    print("\n=== Column Change Report ===")
    print("\nNew columns added from CSV:")
    for col in sorted(new_columns):
        print("   +", col)

    print("\nColumns existing in both CSV and SHP:")
    for col in sorted(shared_columns):
        print("   *", col)

    print("\nFinal columns in merged shapefile:")
    print(sorted(merged_cols))

    # Shapefile has a 10-character field-name limitation.
    print_shapefile_truncation_report(df.columns.tolist(), merged.columns.tolist())

    merged.to_file(output_path, encoding="utf-8")
    print(f"\nSaved merged shapefile to:\n{output_path}")

    return merged


def calculate_median_lst(
    gdf: gpd.GeoDataFrame,
    raster_path: str,
    output_field: str,
    count_field: str = "n_pix_LST"
) -> gpd.GeoDataFrame:
    """Calculate median LST for each polygon using zonal statistics."""
    require_file(raster_path, "LST raster")

    gdf_out = gdf.copy()

    with rasterio.open(raster_path) as src:
        arr = src.read(1)
        transform = src.transform
        nodata = src.nodata

    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)

    # In the provided rasters, 0 outside the mask is treated as invalid.
    arr = np.where(arr == 0, np.nan, arr)

    zs = zonal_stats(
        gdf_out,
        arr,
        affine=transform,
        stats=["median", "count"],
        nodata=np.nan
    )

    gdf_out[output_field] = [z["median"] for z in zs]
    gdf_out[count_field] = [z["count"] for z in zs]

    print(f"Computed {output_field}")
    return gdf_out


def spearman_rho(df: pd.DataFrame, x_col: str, y_col: str, min_n: int = 5) -> Tuple[float, int]:
    """Calculate Spearman rho between two columns after dropping missing values."""
    sub = df[[x_col, y_col]].copy()
    sub[x_col] = pd.to_numeric(sub[x_col], errors="coerce")
    sub[y_col] = pd.to_numeric(sub[y_col], errors="coerce")
    sub = sub.dropna()

    n = len(sub)
    if n < min_n:
        return np.nan, n

    rho = sub[[x_col, y_col]].corr(method="spearman").iloc[0, 1]
    return float(rho), n


def spearman_all_and_by_urbanity(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    group_col: str = STE_COL,
    min_n: int = 5,
    positive_x_only: bool = False
) -> pd.DataFrame:
    """Calculate Spearman rho for all neighborhoods and by urbanity category."""
    work = df.copy()
    work[group_col] = pd.to_numeric(work[group_col], errors="coerce")
    work[x_col] = pd.to_numeric(work[x_col], errors="coerce")
    work[y_col] = pd.to_numeric(work[y_col], errors="coerce")

    work = work.dropna(subset=[group_col, x_col, y_col])
    if positive_x_only:
        work = work[work[x_col] > 0]

    rows = []

    rho_all, n_all = spearman_rho(work, x_col, y_col, min_n=min_n)
    rows.append({
        "group": "ALL",
        "n": n_all,
        "x_variable": x_col,
        "y_variable": y_col,
        "spearman_rho": rho_all
    })

    print(f"\nSpearman rho: {x_col} vs {y_col}")
    print(f"ALL: rho = {rho_all:.3f} (n={n_all})" if not np.isnan(rho_all) else f"ALL: too few samples (n={n_all})")

    for urbanity in sorted(work[group_col].dropna().unique()):
        group_df = work[work[group_col] == urbanity]
        rho_group, n_group = spearman_rho(group_df, x_col, y_col, min_n=min_n)

        rows.append({
            "group": f"ste_mvs={int(urbanity)}",
            "n": n_group,
            "x_variable": x_col,
            "y_variable": y_col,
            "spearman_rho": rho_group
        })

        if np.isnan(rho_group):
            print(f"ste_mvs={int(urbanity)}: too few samples (n={n_group})")
        else:
            print(f"ste_mvs={int(urbanity)}: rho = {rho_group:.3f} (n={n_group})")

    return pd.DataFrame(rows)


# ============================================================
# 3. FULL LST VS REPC ANALYSIS
# ============================================================

def run_repc_lst_analysis(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """Analyze Spearman correlation between REPC and full-neighborhood median LST."""
    print("\n" + "=" * 80)
    print("FULL LST vs REPC ANALYSIS")
    print("=" * 80)

    repc_col = find_existing_column(gdf, REPC_CANDIDATES, "REPC column")

    for col in [COMM_ID_COL, STE_COL, repc_col]:
        if col not in gdf.columns:
            raise ValueError(f"Column '{col}' not found in merged shapefile.")

    gdf_lst = calculate_median_lst(
        gdf=gdf,
        raster_path=FULL_LST_TIF,
        output_field=FULL_LST_FIELD,
        count_field="n_pix_LST_full"
    )

    df_corr = gdf_lst.dropna(subset=[FULL_LST_FIELD, repc_col, STE_COL]).copy()

    print("\nSamples used for REPC-LST correlation:", len(df_corr))
    print("ste_mvs distribution:")
    print(df_corr[STE_COL].value_counts().sort_index())

    results = spearman_all_and_by_urbanity(
        df=df_corr,
        x_col=repc_col,
        y_col=FULL_LST_FIELD,
        group_col=STE_COL,
        min_n=5,
        positive_x_only=False
    )

    output_path = os.path.join(MODEL_DIR, "Spearman_REPC_LST_by_ste_mvs_and_all.csv")
    results.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"\nSaved REPC-LST Spearman results:\n{output_path}")

    group_median = (
        df_corr
        .groupby(STE_COL)[[repc_col, FULL_LST_FIELD]]
        .median()
        .reset_index()
    )

    group_median_output = os.path.join(MODEL_DIR, "Median_REPC_LST_by_ste_mvs.csv")
    group_median.to_csv(group_median_output, index=False, encoding="utf-8-sig")
    print(f"Saved REPC-LST group medians:\n{group_median_output}")

    return results


# ============================================================
# 4. LAND-COVER-SPECIFIC LST VS LANDSCAPE METRICS
# ============================================================

def run_landcover_lst_metric_analysis(gdf: gpd.GeoDataFrame, land_cover: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run LST vs landscape-metric Spearman analysis for one land-cover type."""
    if land_cover not in LAND_COVER_TYPES:
        raise ValueError(f"Unsupported land_cover: {land_cover}. Choose from {LAND_COVER_TYPES}")

    print("\n" + "=" * 80)
    print(f"LAND-COVER LST ANALYSIS: {land_cover}")
    print("=" * 80)

    primary_col = PRIMARY_METRIC[land_cover]
    secondary_col = SECONDARY_METRIC[land_cover]
    lst_field = f"LST_{land_cover}_median"
    raster_path = os.path.join(DATA_DIR, LST_TIF_NAME[land_cover])

    needed_cols = [COMM_ID_COL, STE_COL, primary_col, secondary_col]
    for col in needed_cols:
        if col not in gdf.columns:
            raise ValueError(
                f"Missing column in shapefile for '{land_cover}': {col}. "
                f"Check shapefile field truncation or update PRIMARY_METRIC/SECONDARY_METRIC."
            )

    gdf_lst = calculate_median_lst(
        gdf=gdf,
        raster_path=raster_path,
        output_field=lst_field,
        count_field=f"n_pix_{land_cover}"
    )

    df_base = gdf_lst[
        (~gdf_lst[lst_field].isna()) &
        (gdf_lst[STE_COL].notna())
    ].copy()

    print("Valid samples:", len(df_base))

    # Overall results for primary and secondary metrics.
    overall_rows = []
    grouped_frames = []

    for metric_role, metric_col in [("primary", primary_col), ("secondary", secondary_col)]:
        print(f"\nMetric role: {metric_role}; metric: {metric_col}")

        grouped_result = spearman_all_and_by_urbanity(
            df=df_base,
            x_col=metric_col,
            y_col=lst_field,
            group_col=STE_COL,
            min_n=5,
            positive_x_only=True
        )

        grouped_result.insert(0, "land_cover", land_cover)
        grouped_result.insert(1, "metric_role", metric_role)
        grouped_frames.append(grouped_result)

        all_row = grouped_result[grouped_result["group"] == "ALL"].iloc[0]
        overall_rows.append({
            "land_cover": land_cover,
            "metric_role": metric_role,
            "metric": metric_col,
            "lst_field": lst_field,
            "n": all_row["n"],
            "spearman_rho_all": all_row["spearman_rho"]
        })

    overall_df = pd.DataFrame(overall_rows)
    grouped_df = pd.concat(grouped_frames, ignore_index=True)

    return overall_df, grouped_df


def run_all_landcover_lst_metric_analyses(gdf: gpd.GeoDataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run class-specific LST analysis for all supported land-cover types."""
    overall_outputs = []
    grouped_outputs = []

    for land_cover in LAND_COVER_TYPES:
        overall_df, grouped_df = run_landcover_lst_metric_analysis(gdf, land_cover)
        overall_outputs.append(overall_df)
        grouped_outputs.append(grouped_df)

    overall_summary = pd.concat(overall_outputs, ignore_index=True)
    grouped_summary = pd.concat(grouped_outputs, ignore_index=True)

    overall_output = os.path.join(MODEL_DIR, "LST_landcover_metric_spearman_summary.csv")
    grouped_output = os.path.join(MODEL_DIR, "LST_landcover_metric_spearman_grouped.csv")

    overall_summary.to_csv(overall_output, index=False, encoding="utf-8-sig")
    grouped_summary.to_csv(grouped_output, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("FINAL LST-LANDSCAPE SUMMARY")
    print("=" * 80)
    print(overall_summary)
    print(f"\nSaved overall LST-landscape summary:\n{overall_output}")
    print(f"Saved grouped LST-landscape summary:\n{grouped_output}")

    return overall_summary, grouped_summary


# ============================================================
# 5. RUN ALL ANALYSES
# ============================================================

if __name__ == "__main__":
    merged_gdf = merge_csv_with_shapefile(
        csv_path=CLEANED_CSV,
        shp_path=FILTERED_SHP,
        output_path=MERGED_SHP
    )

    # Reload the saved shapefile so the analysis uses the same truncated field names
    # that will exist on disk.
    merged_gdf = gpd.read_file(MERGED_SHP)

    run_repc_lst_analysis(merged_gdf)
    run_all_landcover_lst_metric_analyses(merged_gdf)

    print("\nALL ADDITIONAL LST ANALYSIS STEPS COMPLETED SUCCESSFULLY!")


C:\Users\Yu Xiaohe\AppData\Local\Temp\ipykernel_13568\402693480.py:36: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should be smooth. However, if you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


PROJECT STRUCTURE READY
BASE_DIR: d:\1st\code\project

Loaded successfully.
Rows in CSV: 10580
Rows in SHP: 10580

=== Loaded CSV Columns ===
['buurtcode', 'PLAND_built', 'PLAND_water', 'PLAND_Dense_Short', 'PLAND_Dense_Tall', 'PLAND_Sparse_Short', 'PLAND_Sparse_Tall', 'PD', 'SHDI', 'AI_built', 'AI_water', 'AI_Dense_Short', 'AI_Dense_Tall', 'AI_Sparse_Short', 'AI_Sparse_Tall', 'IJI_built', 'IJI_water', 'bld_A_pct', 'bev_dich', 'p_1gezw', 'p_bjj2k', 'p_ink_hi', 'ave_floor', 'building_Density', 'g_ele', 'g_gas', 'p_stadsv', 'p_bewndw', 'ec_inten_percapita', 'g_hhgro', 'ste_mvs', 'ec_total']

=== Loaded SHP Columns ===
['buurtcode', 'buurtnaam', 'gemeenteco', 'gemeentena', 'geometry']

=== Merged GeoDataFrame Preview ===
    buurtcode                    buurtnaam gemeenteco gemeentena  \
0  BU00140000             Binnenstad-Noord     GM0014  Groningen   
1  BU00140001              Binnenstad-Zuid     GM0014  Groningen   
2  BU00140002              Binnenstad-Oost     GM0014  Groningen   


C:\Users\Yu Xiaohe\AppData\Local\Temp\ipykernel_13568\402693480.py:201: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  merged.to_file(output_path, encoding="utf-8")



Saved merged shapefile to:
d:\1st\code\project\shp\merged_cleaned_shapefile.shp

FULL LST vs REPC ANALYSIS
Computed LST_median_2022

Samples used for REPC-LST correlation: 10577
ste_mvs distribution:
ste_mvs
1    1397
2    1971
3    1536
4    1609
5    4064
Name: count, dtype: int64

Spearman rho: ec_inten_p vs LST_median_2022
ALL: rho = -0.405 (n=10577)
ste_mvs=1: rho = -0.149 (n=1397)
ste_mvs=2: rho = -0.256 (n=1971)
ste_mvs=3: rho = -0.303 (n=1536)
ste_mvs=4: rho = -0.184 (n=1609)
ste_mvs=5: rho = -0.214 (n=4064)

Saved REPC-LST Spearman results:
d:\1st\code\project\model\Spearman_REPC_LST_by_ste_mvs_and_all.csv
Saved REPC-LST group medians:
d:\1st\code\project\model\Median_REPC_LST_by_ste_mvs.csv

LAND-COVER LST ANALYSIS: water
Computed LST_water_median
Valid samples: 10577

Metric role: primary; metric: AI_water

Spearman rho: AI_water vs LST_water_median
ALL: rho = 0.168 (n=10355)
ste_mvs=1: rho = -0.091 (n=1373)
ste_mvs=2: rho = -0.067 (n=1920)
ste_mvs=3: rho = 0.043 (n=1498)
s